# **Preparing Dependancies**

In [ ]:
!pip install -q diffusers transformers accelerate safetensors ftfy


# **Kriteria 1: Melakukan Image Generation dari Teks (Text-to-Image)**

## **Load Base Pipeline Model**

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
from diffusers import StableDiffusionPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

MODEL_ID = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    safety_checker=None,
).to(device)

print(f"Base pipeline '{MODEL_ID}' loaded on {device}.")


## **Generate Image**

In [ ]:
def generate_simple_image(prompt, negative_prompt, seed):
    """Text-to-image standar: hanya prompt, negative_prompt, dan seed."""
    generator = torch.Generator(device=device).manual_seed(seed)
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
    ).images[0]
    return image


PROMPT = (
    "an astronaut standing on the surface of mars, earth visible in the "
    "background, vector art, flat illustration, minimalist, clean lines, bold colors"
)
NEGATIVE_PROMPT = (
    "photorealistic, realistic, photograph, 3d render, messy, blurry, low "
    "quality, bad art, ugly, sketch, grainy, unfinished, chromatic aberration"
)
SEED = 222

simple_image = generate_simple_image(PROMPT, NEGATIVE_PROMPT, SEED)
simple_image


## **Generate Image with Hyperparameter Configuration**

In [ ]:
def generate_advanced_image(prompt, negative_prompt, seed, guidance_scale=7.5, num_inference_steps=30):
    """Text-to-image dengan parameter tambahan: guidance_scale & num_inference_steps."""
    generator = torch.Generator(device=device).manual_seed(seed)
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    ).images[0]
    return image


advanced_image = generate_advanced_image(
    PROMPT, NEGATIVE_PROMPT, SEED, guidance_scale=7.5, num_inference_steps=30
)
advanced_image


## **Guidance Scale Comparison**

In [ ]:
def show_images_grid(images, titles, cols=2, figsize=(12, 6)):
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    for ax in axes[len(images):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


guidance_low_image = generate_advanced_image(
    PROMPT, NEGATIVE_PROMPT, SEED, guidance_scale=3.0, num_inference_steps=30
)
guidance_high_image = generate_advanced_image(
    PROMPT, NEGATIVE_PROMPT, SEED, guidance_scale=15.0, num_inference_steps=30
)

show_images_grid(
    [guidance_low_image, guidance_high_image],
    ["Guidance Scale = 3.0 (Rendah)", "Guidance Scale = 15.0 (Tinggi)"],
)


### **Guidance Scale Explanation:**

*   **Gambar dengan "Scale" Rendah (3.0):**
*Model lebih longgar mengikuti prompt sehingga hasilnya lebih "kreatif"/acak, komposisi dan detail objek (helm, pakaian astronot, tekstur permukaan Mars) kurang presisi, dan warna cenderung lebih flat/kurang kontras karena model tidak dipaksa kuat mengikuti teks.*

*   **Gambar dengan "Scale" Tinggi (15.0):**
*Gambar jauh lebih patuh terhadap prompt (detail astronot, Bumi di latar belakang, gaya vector art terlihat lebih jelas dan konsisten), kontras dan saturasi warna meningkat, tetapi pada nilai yang terlalu tinggi bisa mulai muncul artefak seperti oversaturation atau distorsi bentuk karena model "memaksakan" kesesuaian dengan prompt secara berlebihan.*


## **Inference Steps Comparison**

In [ ]:
steps_low_image = generate_advanced_image(
    PROMPT, NEGATIVE_PROMPT, SEED, guidance_scale=7.5, num_inference_steps=10
)
steps_high_image = generate_advanced_image(
    PROMPT, NEGATIVE_PROMPT, SEED, guidance_scale=7.5, num_inference_steps=40
)

show_images_grid(
    [steps_low_image, steps_high_image],
    ["Inference Steps = 10 (Rendah)", "Inference Steps = 40 (Tinggi)"],
)


### **Inference Step Explanation:**

*   **Gambar dengan "Step" Rendah (10):**
*Proses denoising belum sepenuhnya konvergen sehingga bentuk objek terlihat kurang tajam, tepi/garis pada ilustrasi vector art tampak kurang rapi, dan berpotensi muncul noise atau artefak kecil karena model belum sempat menghaluskan detail.*

*   **Gambar dengan "Step" Tinggi (40):**
*Hasil jauh lebih halus dan detail, garis vector art lebih bersih dan tajam, noise/artefak nyaris hilang, serta bentuk objek (astronot, Bumi, permukaan Mars) lebih stabil dan konsisten dibanding step rendah. Namun peningkatan step di atas titik tertentu memberi peningkatan kualitas yang semakin kecil (diminishing returns) sementara waktu komputasi terus bertambah.*


## **Batch Inference from One Prompt**

In [ ]:
def generate_batch_images(prompt, negative_prompt, seed, guidance_scale=7.5, num_inference_steps=30, num_images=4):
    """Batch inference: menghasilkan beberapa gambar sekaligus dari satu prompt."""
    generators = [
        torch.Generator(device=device).manual_seed(seed + i) for i in range(num_images)
    ]
    images = pipe(
        prompt=[prompt] * num_images,
        negative_prompt=[negative_prompt] * num_images,
        generator=generators,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    ).images
    return images


def make_grid(images, rows=2, cols=2):
    w, h = images[0].size
    grid = Image.new("RGB", (cols * w, rows * h))
    for idx, img in enumerate(images):
        grid.paste(img, ((idx % cols) * w, (idx // cols) * h))
    return grid


batch_images = generate_batch_images(
    PROMPT, NEGATIVE_PROMPT, SEED, guidance_scale=7.5, num_inference_steps=30, num_images=4
)
batch_grid = make_grid(batch_images, rows=2, cols=2)
batch_grid


## **Load Scheduler**

In [ ]:
from diffusers import (
    EulerAncestralDiscreteScheduler,
    DPMSolverMultistepScheduler,
    DDIMScheduler,
)


def load_scheduler(pipe, scheduler_name):
    """Mengganti algoritma sampling pada pipeline tanpa perlu memuat ulang model."""
    config = pipe.scheduler.config
    if scheduler_name == "Euler A":
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(config)
    elif scheduler_name == "DPM++":
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(config)
    elif scheduler_name == "DDIM":
        pipe.scheduler = DDIMScheduler.from_config(config)
    else:
        raise ValueError(f"Scheduler '{scheduler_name}' tidak dikenali.")
    return pipe


scheduler_images = []
scheduler_titles = []
for scheduler_name in ["Euler A", "DPM++", "DDIM"]:
    pipe = load_scheduler(pipe, scheduler_name)
    img = generate_advanced_image(PROMPT, NEGATIVE_PROMPT, SEED, guidance_scale=7.5, num_inference_steps=30)
    scheduler_images.append(img)
    scheduler_titles.append(scheduler_name)

show_images_grid(scheduler_images, scheduler_titles, cols=3, figsize=(15, 5))

# Kembalikan ke scheduler default agar sel-sel berikutnya tetap konsisten
pipe = load_scheduler(pipe, "Euler A")


### **Scheduler Comparation:**

*   **Gambar dengan "Euler A Scheduler":**
*Menghasilkan gambar dengan variasi/kreativitas cukup tinggi karena sifatnya stokastik (ancestral sampling), style vector art terlihat "hidup" namun bisa sedikit berbeda tiap step count.*

*   **Gambar dengan "DPM++ Scheduler":**
*Konvergen lebih cepat dengan step yang relatif sedikit, detail tajam dan stabil, menjadi salah satu scheduler favorit karena rasio kualitas terhadap kecepatan yang baik.*

*   **Gambar dengan "DDIM Scheduler":**
*Bersifat deterministik (hasil lebih konsisten pada seed yang sama), gambar cenderung lebih halus/smooth tetapi terkadang kurang detail dibanding DPM++ pada jumlah step yang sama.*


# **Kriteria 2: Menyempurnakan Gambar Melalui Image-to-Image**

## **Inpainting**

### **Load Model Inpainting**

In [ ]:
from diffusers import StableDiffusionInpaintPipeline

INPAINT_MODEL_ID = "runwayml/stable-diffusion-inpainting"

inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    INPAINT_MODEL_ID,
    torch_dtype=dtype,
    safety_checker=None,
).to(device)

print(f"Inpainting pipeline '{INPAINT_MODEL_ID}' loaded on {device}.")


### **Manual Masking**

In [ ]:
# Gunakan hasil generate_simple_image sebagai gambar dasar untuk inpainting
base_image_for_inpaint = simple_image.resize((512, 512))


def make_manual_mask(size, box):
    """Membuat mask putih (area yang akan di-edit) pada koordinat box (x0, y0, x1, y1)."""
    mask = Image.new("L", size, 0)
    draw = ImageDraw.Draw(mask)
    draw.rectangle(box, fill=255)
    return mask


# Koordinat box ditentukan melalui trial-and-error di area di samping/bawah astronot
MANUAL_MASK_BOX = (260, 210, 470, 360)
manual_mask = make_manual_mask(base_image_for_inpaint.size, MANUAL_MASK_BOX)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(base_image_for_inpaint)
axes[0].set_title("Gambar Asli")
axes[0].axis("off")
axes[1].imshow(manual_mask, cmap="gray")
axes[1].set_title("Mask Manual (Hardcode)")
axes[1].axis("off")
plt.show()


### **Generate**

In [ ]:
def inpaint_engine(image, mask, prompt, negative_prompt="", seed=9, guidance_scale=7.5, num_inference_steps=30):
    """Menerima image, mask, dan prompt; menjalankan Stable Diffusion Inpainting."""
    generator = torch.Generator(device=device).manual_seed(seed)
    result = inpaint_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=image,
        mask_image=mask,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    ).images[0]
    return result


INPAINT_PROMPT = (
    "a broken satellite with damaged solar panels lying next to the astronaut, "
    "vector art, flat illustration, minimalist, clean lines"
)

manual_inpaint_result = inpaint_engine(
    base_image_for_inpaint, manual_mask, INPAINT_PROMPT, negative_prompt=NEGATIVE_PROMPT, seed=9
)
manual_inpaint_result


## **Inpainting Menggunakan Automasking**

### **load Model Segmentation Untuk Masking**

In [ ]:
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation

SEGMENTATION_MODEL_ID = "CIDAS/clipseg-rd64-refined"

seg_processor = CLIPSegProcessor.from_pretrained(SEGMENTATION_MODEL_ID)
seg_model = CLIPSegForImageSegmentation.from_pretrained(SEGMENTATION_MODEL_ID).to(device)

print(f"Segmentation model '{SEGMENTATION_MODEL_ID}' loaded on {device}.")


### **Masking with Segmentation Model**

In [ ]:
def generate_auto_mask(image, text_prompt, threshold=0.4):
    """Menghasilkan mask otomatis dengan CLIPSeg berdasarkan deskripsi teks area yang ingin di-mask."""
    inputs = seg_processor(text=[text_prompt], images=[image], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = seg_model(**inputs)

    preds = torch.sigmoid(outputs.logits).cpu().numpy()
    preds = (preds - preds.min()) / (preds.max() - preds.min() + 1e-8)

    mask_img = Image.fromarray((preds * 255).astype("uint8")).resize(image.size)
    binary_mask = mask_img.point(lambda p: 255 if p > threshold * 255 else 0)
    return binary_mask


auto_mask = generate_auto_mask(base_image_for_inpaint, "astronaut", threshold=0.4)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(base_image_for_inpaint)
axes[0].set_title("Gambar Asli")
axes[0].axis("off")
axes[1].imshow(auto_mask, cmap="gray")
axes[1].set_title("Mask Otomatis (CLIPSeg)")
axes[1].axis("off")
plt.show()


### **Generate**

In [ ]:
auto_inpaint_result = inpaint_engine(
    base_image_for_inpaint, auto_mask, INPAINT_PROMPT, negative_prompt=NEGATIVE_PROMPT, seed=9
)
auto_inpaint_result


## **Outpainting**

### **Prepare the Canvas**

In [ ]:
def prepare_outpainting(image, direction="right", expand_pixels=256):
    """Memperluas kanvas ke SATU arah (left/right/top/bottom) dan menyiapkan mask-nya."""
    w, h = image.size
    if direction in ("left", "right"):
        new_w, new_h = w + expand_pixels, h
    elif direction in ("top", "bottom"):
        new_w, new_h = w, h + expand_pixels
    else:
        raise ValueError("direction harus salah satu dari: left, right, top, bottom")

    # Safety: resolusi harus kelipatan 8 untuk Stable Diffusion
    new_w -= new_w % 8
    new_h -= new_h % 8

    # Background: gambar asli di-resize & blur agar transisi lebih halus
    bg = image.resize((new_w, new_h), resample=Image.BICUBIC).filter(ImageFilter.GaussianBlur(50))
    canvas = bg.copy()

    if direction == "right":
        paste_pos = (0, 0)
    elif direction == "left":
        paste_pos = (new_w - w, 0)
    elif direction == "bottom":
        paste_pos = (0, 0)
    else:  # top
        paste_pos = (0, new_h - h)

    canvas.paste(image, paste_pos)

    # Mask: putih = area yang akan digenerate ulang, hitam = area asli yang dipertahankan
    mask = Image.new("L", (new_w, new_h), 255)
    keep_area = Image.new("L", image.size, 0)
    mask.paste(keep_area, paste_pos)

    return canvas, mask


outpaint_canvas, outpaint_mask = prepare_outpainting(
    manual_inpaint_result, direction="right", expand_pixels=256
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(outpaint_canvas)
axes[0].set_title("Canvas Diperluas (kanan)")
axes[0].axis("off")
axes[1].imshow(outpaint_mask, cmap="gray")
axes[1].set_title("Mask Outpainting")
axes[1].axis("off")
plt.show()


### **Generate**

In [ ]:
OUTPAINT_PROMPT = (
    "wide angle view of an astronaut on mars next to a broken satellite, earth "
    "in the background, expansive martian landscape, vector art, flat illustration"
)

outpaint_result = inpaint_engine(
    outpaint_canvas, outpaint_mask, OUTPAINT_PROMPT, negative_prompt=NEGATIVE_PROMPT,
    seed=9, num_inference_steps=30,
)
outpaint_result


## **Outpainting Zoom Out**

### **Prepare Canvas for Zoom Out**

In [ ]:
def prepare_zoom_out(image, expand_pixels=128):
    """Memperluas kanvas ke SEMUA arah sekaligus (dipakai berulang untuk efek Zoom Out)."""
    w, h = image.size
    new_w, new_h = w + expand_pixels * 2, h + expand_pixels * 2
    new_w -= new_w % 8
    new_h -= new_h % 8

    bg = image.resize((new_w, new_h), resample=Image.BICUBIC).filter(ImageFilter.GaussianBlur(50))
    canvas = bg.copy()

    paste_x, paste_y = (new_w - w) // 2, (new_h - h) // 2
    canvas.paste(image, (paste_x, paste_y))

    mask = Image.new("L", (new_w, new_h), 255)
    keep_area = Image.new("L", image.size, 0)
    mask.paste(keep_area, (paste_x, paste_y))

    return canvas, mask


def zoom_out(image, prompt, negative_prompt="", steps_count=2, expand_pixels=128, seed=9):
    """Menjalankan outpainting berkali-kali secara bertahap ke berbagai arah (Zoom Out)."""
    current = image
    history = [current]
    for i in range(steps_count):
        canvas, mask = prepare_zoom_out(current, expand_pixels=expand_pixels)
        current = inpaint_engine(
            canvas, mask, prompt, negative_prompt=negative_prompt,
            seed=seed + i, num_inference_steps=30,
        )
        history.append(current)
    return current, history


ZOOM_PROMPT = (
    "wide angle view of an astronaut on mars next to a broken satellite, earth "
    "in the background, expansive martian landscape, vector art, flat illustration"
)


### **Generate**

In [ ]:
zoom_result, zoom_history = zoom_out(
    manual_inpaint_result, ZOOM_PROMPT, negative_prompt=NEGATIVE_PROMPT,
    steps_count=2, expand_pixels=128, seed=9,
)

fig, axes = plt.subplots(1, len(zoom_history), figsize=(5 * len(zoom_history), 5))
for ax, img, step in zip(axes, zoom_history, range(len(zoom_history))):
    ax.imshow(img)
    ax.set_title(f"Zoom Step {step}")
    ax.axis("off")
plt.show()


## **Base + Refiner Image Generation**

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline

refiner_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    safety_checker=None,
).to(device)


def generate_with_refiner(prompt, negative_prompt, seed, guidance_scale=7.5, num_inference_steps=40):
    """Two-Stage Generation (Base + Refiner).

    Stage 1: pipeline Base menghasilkan latent awal (berhenti di 80% proses denoising).
    Stage 2: latent tersebut dioper ke pipeline Img2Img untuk menyempurnakan 20% sisanya.
    """
    generator = torch.Generator(device=device).manual_seed(seed)
    base_latents = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        denoising_end=0.8,
        output_type="latent",
    ).images

    generator = torch.Generator(device=device).manual_seed(seed)
    refined_image = refiner_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=base_latents,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        denoising_start=0.8,
    ).images[0]

    return refined_image


refined_result = generate_with_refiner(PROMPT, NEGATIVE_PROMPT, SEED)
refined_result
